In [1]:
import random
from textblob import TextBlob
import numpy as np
from IPython.display import Audio, display

In [2]:
# 1. Base Configurations
RAGAS = {
    "Bilawal": {"notes": ["Sa", "Re", "Ga", "Ma", "Pa", "Dha", "Ni"], "mood": "happy"},
    "Shivaranjani": {"notes": ["Sa", "Re", "ga", "Pa", "Dha"], "mood": "sad"},
    "Yaman": {"notes": ["Sa", "Re", "Ga", "ma", "Pa", "Dha", "Ni"], "mood": "peaceful"}
}

FREQUENCIES = {
    'Sa': 261.63, 're': 277.18, 'Re': 293.66, 'ga': 311.13, 'Ga': 329.63,
    'Ma': 349.23, 'ma': 369.99, 'Pa': 392.00, 'dha': 415.30, 'Dha': 440.00,
    'ni': 466.16, 'Ni': 493.88,
}

def analyze_sentiment(lyrics):
    analysis = TextBlob(lyrics)
    polarity = analysis.sentiment.polarity
    if polarity > 0.2: return "happy", "Bilawal"
    elif polarity < -0.2: return "sad", "Shivaranjani"
    else: return "peaceful", "Yaman"

def generate_rhythmic_tune(lyrics):
    """Generates a Sargam sequence with calculated pitch and duration per word."""
    mood, raga_name = analyze_sentiment(lyrics)
    allowed_notes = RAGAS[raga_name]["notes"]
    max_index = len(allowed_notes) - 1
    
    words = lyrics.split()
    tune = []
    current_index = 0
    
    print(f"--- Rhythmic Tune Generation ---")
    print(f"Detected Mood: {mood.capitalize()} | Raga: {raga_name}\n")
    print(f"{'WORD'.ljust(15)} | {'NOTE'.ljust(5)} | DURATION")
    print("-" * 35)
    
    for i, word in enumerate(words):
        clean_word = word.lower().strip(".,!?")
        if not clean_word: continue
        
        word_length = len(clean_word)
        
        # --- NEW RHYTHM LOGIC ---
        # Map word length to musical note durations (in seconds)
        if word_length <= 3:
            duration = 0.25  # Quick passing note (Eighth note)
            step = 1
        elif word_length <= 5:
            duration = 0.50  # Standard beat (Quarter note)
            step = 2
        else:
            duration = 1.00  # Long held note (Half note)
            step = 3
            
        # Add a special rule: The very last word of the sentence should ring out
        if i == len(words) - 1:
            duration = 1.50 

        # --- PITCH LOGIC (Same as before) ---
        starts_with_vowel = clean_word[0] in "aeiou"
        direction = 1 if starts_with_vowel else -1
        
        current_index += (direction * step)
        
        # Boundary logic (Bounce)
        if current_index > max_index:
            current_index = max_index - (current_index - max_index)
        elif current_index < 0:
            current_index = abs(current_index)
            
        current_index = max(0, min(current_index, max_index))
        note = allowed_notes[current_index]
        
        # We now store a 3-part tuple: (Word, Note, Duration)
        tune.append((word, note, duration))
        
        print(f"{word.ljust(15)} | {note.ljust(5)} | {duration}s")
        
    return tune

def play_dynamic_sargam(tune, sample_rate=44100):
    """Generates a continuous audio wave accounting for varying note durations."""
    print("\nCompiling dynamic audio sequence...")
    
    audio_sequence = []
    
    # 1. Build an array of frequencies that changes based on the calculated durations
    for word, note, duration in tune:
        freq = FREQUENCIES.get(note, FREQUENCIES['Sa'])
        samples_for_this_note = int(duration * sample_rate)
        
        # Create a block of frequency values for exactly this duration
        freq_block = np.full(samples_for_this_note, freq)
        audio_sequence.append(freq_block)
        
    # Snap them all together into one timeline
    freq_array = np.concatenate(audio_sequence)
    total_samples = len(freq_array)
    
    # 2. Apply "Meend" (Smooth the jumps between frequencies)
    glide_window = int(sample_rate * 0.1) 
    padded_freq = np.pad(freq_array, (glide_window, glide_window), mode='edge')
    smooth_freq = np.convolve(padded_freq, np.ones(glide_window)/glide_window, mode='valid')
    smooth_freq = smooth_freq[:total_samples] 

    # 3. Calculate Continuous Phase to draw the sine wave
    phase = np.cumsum(smooth_freq) * 2 * np.pi / sample_rate
    final_audio = np.sin(phase)
    
    # 4. Apply Fade In/Out
    fade_len = int(sample_rate * 0.2)
    final_audio[:fade_len] *= np.linspace(0, 1, fade_len)
    final_audio[-fade_len:] *= np.linspace(1, 0, fade_len)
    
    print("Ready to play!")
    display(Audio(final_audio, rate=sample_rate))

In [4]:
# --- Let's test the dynamic rhythm! ---
# lyrics_input = "The sun is shining bright and my heart is full of joy today"
lyrics_input = "Neither the body nor the mind but I am the atman, saying thus you can become one with the paramatman"
rhythmic_tune = generate_rhythmic_tune(lyrics_input)

play_dynamic_sargam(rhythmic_tune)

--- Rhythmic Tune Generation ---
Detected Mood: Peaceful | Raga: Yaman

WORD            | NOTE  | DURATION
-----------------------------------
Neither         | ma    | 1.0s
the             | Ga    | 0.25s
body            | Sa    | 0.5s
nor             | Re    | 0.25s
the             | Sa    | 0.25s
mind            | Ga    | 0.5s
but             | Re    | 0.25s
I               | Ga    | 0.25s
am              | ma    | 0.25s
the             | Ga    | 0.25s
atman,          | Pa    | 0.5s
saying          | Re    | 1.0s
thus            | Re    | 0.5s
you             | Sa    | 0.25s
can             | Re    | 0.25s
become          | Ga    | 1.0s
one             | ma    | 0.25s
with            | Re    | 0.5s
the             | Sa    | 0.25s
paramatman      | ma    | 1.5s

Compiling dynamic audio sequence...
Ready to play!
